# Gerçek Learning-Rate Pilotu — 60 Koşu

Bu notebook tez önerisindeki kontrollü karşılaştırma protokolünün hiperparametre hazırlık aşamasıdır. 3 temsilci aktivasyon × 5 optimizer × 4 LR = **60 koşu**, her biri **30 epoch**.

Notebook Colab oturumları arasında yeniden çalıştırılabilir. Tamamlanan koşular atlanır; yarım kalan koşu checkpointten devam eder. Test seti kullanılmaz.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


## Güncel public GitHub sürümünü al


In [ ]:
import io, os, shutil, tarfile, requests
from pathlib import Path

OWNER='erdemci07'; REPO='derin-ogrenme-egitim-dinamikleri'; BRANCH='main'
repo_dir=Path('/content/derin-ogrenme-egitim-dinamikleri')
extract_root=Path('/content/_tez_repo_extract')
shutil.rmtree(repo_dir, ignore_errors=True)
shutil.rmtree(extract_root, ignore_errors=True)
extract_root.mkdir(parents=True, exist_ok=True)
r=requests.get(f'https://api.github.com/repos/{OWNER}/{REPO}/commits/{BRANCH}', timeout=60)
r.raise_for_status(); commit_sha=r.json()['sha']
a=requests.get(f'https://api.github.com/repos/{OWNER}/{REPO}/tarball/{commit_sha}', timeout=120)
a.raise_for_status()
with tarfile.open(fileobj=io.BytesIO(a.content), mode='r:gz') as tar:
    tar.extractall(extract_root)
kokler=[p for p in extract_root.iterdir() if p.is_dir()]
if len(kokler)!=1:
    raise RuntimeError('GitHub arşiv yapısı beklenenden farklı.')
shutil.move(str(kokler[0]), str(repo_dir)); os.chdir(repo_dir)
os.environ['TEZ_GIT_COMMIT']=commit_sha
print('Repo hazır:',commit_sha)


## Google Drive kalıcı veri ve sonuç yolları


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_KOK=Path('/content/drive/MyDrive')
DRIVE_VERI=DRIVE_KOK/'tez_datasets'
DRIVE_PROJE=DRIVE_KOK/'tez_deney_sonuclari'
DRIVE_AYAR=DRIVE_PROJE/'ortak_ayarlar'
DRIVE_PILOT=DRIVE_PROJE/'lr_pilotu'
DRIVE_ANALIZ=DRIVE_PROJE/'lr_pilotu_analiz'
if not DRIVE_VERI.is_dir():
    raise FileNotFoundError(f'Veri klasörü bulunamadı: {DRIVE_VERI}')
DRIVE_AYAR.mkdir(parents=True, exist_ok=True)
DRIVE_PILOT.mkdir(parents=True, exist_ok=True)
DRIVE_ANALIZ.mkdir(parents=True, exist_ok=True)
SPLIT_DOSYASI=DRIVE_AYAR/'cifar10_split.json'
STATS_DOSYASI=DRIVE_AYAR/'cifar10_stats.json'
os.environ['TEZ_VERI_KLASORU']=str(DRIVE_VERI)
os.environ['TEZ_SPLIT_DOSYASI']=str(SPLIT_DOSYASI)
os.environ['TEZ_STATS_DOSYASI']=str(STATS_DOSYASI)
os.environ['TEZ_CIKTI_KLASORU']=str(DRIVE_PILOT)
print('Pilot çıktıları:',DRIVE_PILOT)


## CUDA ve bağımlılıklar


In [ ]:
subprocess.run(['python','-m','pip','install','-q','-r','requirements_colab.txt'], check=True)
import torch
print('PyTorch:',torch.__version__)
print('CUDA:',torch.cuda.is_available())
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')
if not torch.cuda.is_available():
    raise RuntimeError('T4/CUDA aktif değil.')


## Pilotu çalıştır / devam ettir

Bir Colab oturumunda en fazla yaklaşık **210 dakika** yeni koşu başlatılır. Süre dolunca temiz biçimde durur. Sonraki T4 oturumunda bu notebook yeniden `Run all` yapıldığında tamamlanan koşular atlanır ve çalışma devam eder.


In [ ]:
import importlib
import calistir_lr_pilotu as lr
importlib.reload(lr)
lr.ana(['--calistir','--oturum-dakika','210'])


## İlerlemeyi göster


In [ ]:
tamamlanan=list(DRIVE_PILOT.glob('*/tamamlandi.json'))
kararsiz=list(DRIVE_PILOT.glob('*/kararsiz.json'))
checkpoint=list(DRIVE_PILOT.glob('*/checkpoint.pt'))
print('Tamamlanan:',len(tamamlanan),'/60')
print('Kararsız:',len(kararsiz))
print('Yarım/checkpoint:',len(checkpoint))
if len(tamamlanan)+len(kararsiz)<60:
    print('Pilot tamamlanmadı. Yeni T4 oturumunda aynı notebooku yeniden çalıştırın.')


## 60 koşu tamamlandıysa LR seçimini yap

Bu hücre pilot tamamlanmadan çalıştırılırsa bilinçli olarak hata verir. Seçim yalnız validation verisiyle ve önceden tanımlı kuralla yapılır; test seti kullanılmaz.


In [ ]:
if len(tamamlanan)+len(kararsiz)==60:
    import analiz_lr_pilotu as analiz
    importlib.reload(analiz)
    analiz.ana(['--pilot-klasoru',str(DRIVE_PILOT),'--cikti-klasoru',str(DRIVE_ANALIZ)])
else:
    print('Henüz seçim analizi yapılmadı.')
